In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import pandas as pd

# 读取原始数据
df = pd.read_csv('T1.csv')

# 确保时间列是datetime类型
df['Time'] = pd.to_datetime(df['Time'], 
                                    format='mixed', 
                                    dayfirst=True,
                                    errors='coerce')  # 无法解析的设为NaT

# 创建1分钟间隔的时间序列
start_time = df['Time'].min()
end_time = df['Time'].max()
time_1min = pd.date_range(start=start_time, end=end_time, freq='1min')

# 线性插值（所有站点同时插值）
df_indexed = df.set_index('Time')
df_1min_indexed = df_indexed.reindex(time_1min)
df_1min_indexed[['WindSpeed']] = df_indexed[['WindSpeed']].reindex(time_1min).interpolate(method='linear')

# 重置索引并保存为CSV（无行数限制）
df_1min_inter = df_1min_indexed.reset_index().rename(columns={'index': 'Time'})

print(f"插值完成！共生成 {len(df_1min_inter):,} 条1分钟数据，已保存为CSV。")

In [ ]:
df_original = df
# -------------------------- 2. 配置波动参数 --------------------------
sites = ['WindSpeed']
noise_params = {
    'white_noise_std': 0.05,       # 高频白噪声标准差（占原始风速的比例，如0.2表示20%波动）
    'diurnal_amplitude': 0.05,     # 日周期波动幅度（原始风速的10%）
    'clip_min': 0                 # 风速最小值（避免负数）
}

# -------------------------- 3. 生成复合随机波动 --------------------------
def generate_wind_noise(df, sites, params):
    """
    生成符合风速特性的复合随机波动：
    - 高频白噪声（短时间尺度随机起伏）
    - 低频日周期波动（缓慢的日变化）
    """
    n = len(df)
    t = np.arange(n)  # 时间索引（分钟）
    
    # 1. 高频白噪声（每个站点独立生成）
    white_noise = np.zeros((n, len(sites)))
    for i, site in enumerate(sites):
        # 白噪声标准差：基于原始数据的标准差调整
        site_std = df_original[site].std()
        white_noise[:, i] = norm.rvs(
            loc=0,
            scale=site_std * params['white_noise_std'],
            size=n
        )
    
    # 2. 低频日周期波动（所有站点共享日趋势）
    diurnal_cycle = np.sin(2 * np.pi * t / 1440)  # 1440分钟=1天，周期为1天
    diurnal_noise = diurnal_cycle.reshape(-1, 1) * params['diurnal_amplitude']
    
    # 3. 合成总波动（高频+低频）
    total_noise = white_noise + diurnal_noise

    for i in range(len(total_noise)):
        if i % 10 == 0:
            total_noise[i]=0
        else:
            None
    return total_noise

# 生成波动
noise = generate_wind_noise(df_1min_inter, sites, noise_params)

# -------------------------- 4. 叠加波动并修正数据 --------------------------
df_noisy = df_1min_inter.copy()
for i, site in enumerate(sites):
    # 叠加波动
    df_noisy[site] = df_1min_inter[site] + noise[:, i]
df_noisy.head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# 读取CSV文件
df = df_noisy.copy()

# 打印前5行，检查数据格式
print("数据前5行：")
print(df.head())

# 将第一列转换为时间序列
df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0], errors='coerce')

# 检查时间转换是否成功
if df.iloc[:, 0].isna().any():
    print("警告：部分时间数据无法解析，可能格式不一致！")

# 提取时间和数据
time = df.iloc[:, 0]
data = pd.to_numeric(df.iloc[:, 1], errors='coerce')  # 确保第二列是数值

# 检查数据是否有效
if data.isna().any():
    print("警告：数据列中存在无效值（非数值），已转换为NaN！")

# 计算布林通道（周期=100，k=2）
window = 500
k = 1
# 中轨：100周期移动平均
middle_band = data.rolling(window=window).mean()
# 标准差
std = data.rolling(window=window).std()
# 上轨和下轨
upper_band = middle_band + k * std
lower_band = middle_band - k * std

# 绘制折线图和布林通道
plt.figure(figsize=(12, 6))  # 增大画布，适应大数据
# 绘制原始数据
plt.plot(time, data, color='blue', label='Wind Data', linewidth=1)
# 绘制布林通道
plt.plot(time, middle_band, color='orange', label='Middle Band (SMA 100)', linewidth=1)
plt.plot(time, upper_band, color='green', label='Upper Band', linewidth=1)
plt.plot(time, lower_band, color='red', label='Lower Band', linewidth=1)
# 填充上轨和下轨之间的区域
plt.fill_between(time, lower_band, upper_band, color='gray', alpha=0.2, label='Bollinger Band')

# 设置Y轴标签和标题
plt.ylabel('Data Value')
plt.title('1-Minute Wind Data with Bollinger Bands (Period=100)')
plt.legend()
plt.grid(True)

# 移除X轴标签和刻度
plt.gca().set_xticks([])  # 不显示X轴刻度
plt.gca().set_xlabel('')  # 移除X轴标题

# 优化布局
plt.tight_layout()

# 保存图片（防止显示卡顿）
plt.savefig('wind_plot_with_bollinger.png', dpi=100)
print("图表已保存为 wind_plot_with_bollinger.png，请检查当前目录！")

# 显示图形（如果环境支持）
plt.show()


In [ ]:
# 读取原始数据
df = pd.read_csv('T1.csv')

# 确保时间列是datetime类型
df['Time'] = pd.to_datetime(df['Time'], 
                                    format='mixed', 
                                    dayfirst=True,
                                    errors='coerce')  # 无法解析的设为NaT
# 统计数值大于上轨或小于下轨的点
outliers = data[(data > upper_band) | (data < lower_band)]
outlier_count = len(outliers)
outlier_percentage = (outlier_count / len(data)) * 100
divisible_by_10 = outliers[outliers.index % 10 == 0]
index_list = divisible_by_10.index / 10
df['index_flag'] = df.index.isin(index_list).astype(int)
df.to_excel('df_T1.xlsx')

In [ ]:
print(f"标志分布: 0-{np.sum(df.index_flag == 0)}个, 1-{np.sum(df.index_flag == 1)}个")